# NB149: The Living Cascade — Studying the System, Not Its Shadows

**Goal**: Understand the cascade ODE as a dynamical system. Stop projecting it onto algebraic patterns and instead study what the 210 branches actually DO — how they organize, interfere, wrap, and produce the structure we see.

**Motivation**: NB148 showed that the CRT eigenstates are NOT the physical states — the "change of basis" from CRT modes to SM fermions IS the dynamics. This means the fermion map cannot be completed algebraically. It must be extracted from the simulation.

**Approach**: 
1. Run the full cascade (JAX, all 210 branches, T = P₄ + 1 = 211)
2. At each coprime crossing, examine the FULL branch distribution — not sector averages, but all 210 R_k values
3. Study how the branches cluster, wrap, and interfere at each level
4. Look for the natural basis in which the dynamics organize the states
5. Let the system tell us what the fermions ARE, rather than imposing an algebraic assignment

The cascade already produces all the mass predictions. But we've always PROJECTED it into the CRT measurement basis to extract CP ratios. What happens if we look at the raw dynamics without projection?

## S0: The Full Cascade — All 210 Branches at All 48 Crossings

First: integrate the complete system and examine the raw output without any CRT projection. For each of the 210 branches, we get R_k(ci) at each of the 48 coprime crossing times, at each of the 4 cascade levels.

That's 210 × 48 × 4 = 40,320 numbers. This is the complete dynamical information of one primorial window. Everything — masses, mixing angles, gauge couplings — is encoded in these 40,320 numbers. Let's look at them.

In [3]:
# ── S0: Full cascade integration and raw data extraction ──

import sys, os, time, numpy as np
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

from solenoid_algebra import SA, KAPPA, EPSILON, OMEGA, CP_PAIRS
from solenoid_system import SolenoidSystem
from solenoid_jax import warmup as jax_warmup, detect_device

print('S0: THE FULL CASCADE — RAW DYNAMICAL DATA')
print('='*70)

# Setup
sys0 = SolenoidSystem()
primes = SA.primes
P4 = SA.P
cis = SA.coprime_indices(P4)
a3, a5, a7 = SA.sector_labels(cis)
all_branches = sys0.all_branches()

print(f'Solenoid: primes = {primes}, P4 = {P4}')
print(f'Branches: {len(all_branches)} = {"×".join(str(p) for p in primes)}')
print(f'Coprime crossings: {len(cis)} in window-0 [1, {P4})')
print(f'Cascade levels: 4 (R0, R1, R2, R3)')
print(f'Total data points: {len(all_branches)} × {len(cis)} × 4 = {len(all_branches)*len(cis)*4}')

# Integrate
print(f'\nJAX device: {detect_device()}')
t0 = time.time()
jax_warmup()
print(f'JAX warmup: {time.time()-t0:.1f}s')

t_cross = cis.astype(float)
T_integ = float(P4 + 1)

t0 = time.time()
res = sys0.integrate_all_branches(all_branches, t_cross, T_integ, backend='jax')
dt = time.time() - t0
print(f'Integration: {dt:.1f}s for {len(all_branches)} branches')

# Extract the full data array: shape (210, 48, 4)
R_all = np.zeros((len(all_branches), len(cis), 4))
for i, br in enumerate(all_branches):
    R_all[i] = res[br]  # shape (48, 4)

print(f'\nRaw data shape: {R_all.shape} = (branches, crossings, levels)')
print(f'Total values: {R_all.size}')

# Basic statistics
print(f'\nPer-level statistics (across all branches and crossings):')
for k in range(4):
    vals = R_all[:, :, k].flatten()
    print(f'  R{k}: mean={vals.mean():+.4f}, std={vals.std():.4f}, '
          f'min={vals.min():.4f}, max={vals.max():.4f}')

# Now: the branch structure. Each branch is (j1, j2, j3, j4).
# At a given crossing ci, how do the 210 branches organize?

# The NB103-104 decomposition: R_k = R_k_ss + 2π j_{k+1} exp(-κ ci)
# This means at each crossing, R_k has a DETERMINISTIC part (R_k_ss, depends on 
# j_1..j_k) and a TRANSIENT part (2π j_{k+1} exp(-κ ci), depends on j_{k+1}).

# Let's verify this at a few crossings and see the structure
print(f'\n\nBRANCH ORGANIZATION AT PHYSICAL CROSSINGS:')
phys_crossings = {'QUARK_g1': 11, 'LEPTON_g1': 31, 'LEPTON_g2': 61, 'QUARK_g2': 191}

for name, ci in phys_crossings.items():
    ci_idx = np.where(cis == ci)[0][0]
    
    print(f'\n  {name} (ci={ci}):')
    for k in range(4):
        R_at_ci = R_all[:, ci_idx, k]
        R_wrapped = np.mod(R_at_ci, 2*np.pi)
        R_wrapped[R_wrapped > np.pi] -= 2*np.pi
        
        # How many distinct clusters? Use the j_{k+1} structure
        # The branches should cluster by j_{k+1} value
        n_unique = len(set(round(r, 2) for r in R_wrapped))
        
        print(f'    R{k}: mean={R_at_ci.mean():+.4f}, std={R_at_ci.std():.4f}, '
              f'wrapped range=[{R_wrapped.min():.3f}, {R_wrapped.max():.3f}], '
              f'~{n_unique} clusters')

# The key observation from NB103-104: at each crossing, the branches
# organize by their j_{k+1} value at level k. At R3 (the mass level),
# the branches cluster into p4 = 7 groups by j4 ∈ {0,...,6}.
# Within each j4 group, the 30 branches (with different j1,j2,j3) 
# have the SAME R3 transient but different R3_ss.

print(f'\n\nR3 AT QUARK g1 (ci=11) BY j4:')
ci_idx = np.where(cis == 11)[0][0]
for j4 in range(primes[3]):
    j4_branches = [i for i, br in enumerate(all_branches) if br[3] == j4]
    R3_vals = R_all[j4_branches, ci_idx, 3]
    R3_wrapped = np.mod(R3_vals, 2*np.pi)
    R3_wrapped[R3_wrapped > np.pi] -= 2*np.pi
    transient = 2*np.pi * j4 * np.exp(-KAPPA * 11)
    print(f'  j4={j4}: {len(j4_branches)} branches, R3 mean={R3_vals.mean():.4f}, '
          f'transient={transient:.4f}, R3_ss mean={R3_vals.mean()-transient:.4f}, '
          f'wrapped RMS={np.sqrt(np.mean(R3_wrapped**2)):.4f}')

S0: THE FULL CASCADE — RAW DYNAMICAL DATA
Solenoid: primes = [2, 3, 5, 7], P4 = 210
Branches: 210 = 2×3×5×7
Coprime crossings: 48 in window-0 [1, 210)
Cascade levels: 4 (R0, R1, R2, R3)
Total data points: 210 × 48 × 4 = 40320

JAX device: CPU (1 device(s))
JAX warmup: 2.1s
  JAX [CPU (1 device(s))]: 210 branches, 48 eval pts, T=211.0 — 2.90s
Integration: 2.9s for 210 branches

Raw data shape: (210, 48, 4) = (branches, crossings, levels)
Total values: 40320

Per-level statistics (across all branches and crossings):
  R0: mean=+0.1972, std=0.7625, min=-0.0110, max=5.8635
  R1: mean=+0.5534, std=1.4799, min=0.0274, max=11.8864
  R2: mean=+0.9802, std=2.7571, min=-0.0438, max=23.7031
  R3: mean=+1.4669, std=3.9472, min=-0.3024, max=35.5004


BRANCH ORGANIZATION AT PHYSICAL CROSSINGS:

  QUARK_g1 (ci=11):
    R0: mean=+1.4647, std=1.4706, wrapped range=[-0.006, 2.935], ~2 clusters
    R1: mean=+3.5156, std=2.4608, wrapped range=[-2.230, 2.978], ~6 clusters
    R2: mean=+6.6514, std=4.2059, 